<a href="https://colab.research.google.com/github/isparshgirics24-glitch/AI-LAB/blob/main/ForwardChaining04/11/2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import itertools
import re


def parse_atom(s):
    s = s.strip()
    match = re.match(r"(\w+)\((.*)\)", s)
    if not match:
        return (s,)
    pred = match.group(1)
    args = [arg.strip() for arg in match.group(2).split(",")]
    return (pred, *args)


def format_atom(atom):
    return f"{atom[0]}({', '.join(atom[1:])})"


def is_variable(x):
    return isinstance(x, str) and len(x) > 0 and x[0].islower()


def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    elif is_variable(x) and x in theta:
        return unify(var, theta[x], theta)
    else:
        new_theta = theta.copy()
        new_theta[var] = x
        return new_theta


def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if theta is False:
        return None
    elif x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, (tuple, list)) and isinstance(y, (tuple, list)):
        if len(x) != len(y):
            return None
        for xi, yi in zip(x, y):
            theta = unify(xi, yi, theta)
            if theta is None:
                return None
        return theta
    return None


def subst(theta, expr):
    if is_variable(expr):
        return theta.get(expr, expr)
    elif isinstance(expr, tuple):
        return tuple(subst(theta, arg) for arg in expr)
    elif isinstance(expr, list):
        return [subst(theta, arg) for arg in expr]
    return expr


facts = {"American(Robert)", "Enemy(A, America)", "Missile(T1)", "Owns(A, T1)"}

rules = [
    ("Enemy(x, America)", "Hostile(x)"),
    ("Missile(x)", "Weapon(x)"),
    ("Missile(x) ∧ Owns(A, x)", "Sells(Robert, x, A)"),
    ("American(p) ∧ Weapon(q) ∧ Sells(p, q, r) ∧ Hostile(r)", "Criminal(p)"),
]

kb_facts = {parse_atom(f) for f in facts}
parsed_rules = []
for premise_str, conclusion_str in rules:
    premises = [parse_atom(p) for p in premise_str.split("∧")]
    conclusion = parse_atom(conclusion_str)
    parsed_rules.append((premises, conclusion))

step = 1
while True:
    new_facts = set()

    for premises, conclusion in parsed_rules:
        k = len(premises)

        for fact_combo in itertools.product(kb_facts, repeat=k):
            theta = {}
            valid = True

            for p, f in zip(premises, fact_combo):
                theta = unify(p, f, theta)
                if theta is None:
                    valid = False
                    break

            if valid and theta is not None:
                inferred = subst(theta, conclusion)

                if inferred not in kb_facts and inferred not in new_facts:
                    ground_premises = [
                        format_atom(subst(theta, p)) for p in premises
                    ]
                    premise_text = " ∧ ".join(ground_premises)
                    conclusion_text = format_atom(inferred)

                    print(f"Step {step}: {premise_text} → {conclusion_text}")
                    new_facts.add(inferred)
                    step += 1

    if not new_facts:
        break

    kb_facts |= new_facts

print("\nFinal Facts:")
for fact in sorted(format_atom(f) for f in kb_facts):
    print(fact)

Step 1: Enemy(A, America) → Hostile(A)
Step 2: Missile(T1) → Weapon(T1)
Step 3: Missile(T1) ∧ Owns(A, T1) → Sells(Robert, T1, A)
Step 4: American(Robert) ∧ Weapon(T1) ∧ Sells(Robert, T1, A) ∧ Hostile(A) → Criminal(Robert)

Final Facts:
American(Robert)
Criminal(Robert)
Enemy(A, America)
Hostile(A)
Missile(T1)
Owns(A, T1)
Sells(Robert, T1, A)
Weapon(T1)
